# 08 — Grad-CAM on M2 (Dhanunjay Rahul Varma)

**Question.** Where on the log-mel spectrogram does M2 look when it calls a
clip synthetic? Two concrete hypotheses, both stated before the run:
- **E2 says the band just below Nyquist matters** (a 7 kHz low-pass cost M2
  5 points, and M1 everything). If so, Grad-CAM mass should be
  over-represented in the top mel rows.
- **Afchar et al. (2025, ref. 9)** predict that the deconvolution /
  upsampling layers in generators leave **systematic spectral peaks**: fixed
  frequencies, present all through the clip. If M2 uses them, the
  time-averaged CAM should show narrow horizontal stripes at the same
  frequencies across clips of a generator, rather than a smooth profile.

**Method.** Grad-CAM (Selvaraju et al. 2017) on the last conv block of M2,
before its final max-pool (a 16 × 125 frequency × time map; each cell covers
8 mel bins × 8 frames = 80 ms). The
trained checkpoint from `fmc-m2-mels` is used unchanged, on all 1,604 test
clips, fp32 (on CPU). The class score is the single logit (positive = synthetic).
M2's global average pooling (§6) is exactly what makes this valid: the
logit is a linear function of the pooled last feature map.

In [ ]:
import os, glob, json
import numpy as np, pandas as pd, torch, torch.nn as nn, librosa
import matplotlib.pyplot as plt

# Grad-CAM needs no GPU: one forward + backward per clip of a 1.2 M-parameter
# CNN. Running on CPU keeps the account's two GPU slots free for M4.
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(os.cpu_count())
print(torch.__version__, dev, os.cpu_count(), 'cpus')

def find_input(name, marker):
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{name}/**/{marker}', recursive=True)
    assert hits, f'{name}/{marker} not found'
    return os.path.dirname(hits[0])

MELS_DIR = find_input('fmc-m2-mels', 'mels.npy')
OUT = '/kaggle/working'
sub    = pd.read_csv(f'{MELS_DIR}/subset.csv', dtype={'ytid': str})
M      = np.load(f'{MELS_DIR}/mels.npy', mmap_mode='r')
labels = np.load(f'{MELS_DIR}/labels.npy')
idx = {s: np.where((sub.split == s).values)[0] for s in ['train', 'test']}
te = idx['test']; y_te = labels[te]; gen_te = sub.generator.values[te]
tr_mean = float(M[idx['train']].astype(np.float32).mean())
tr_std  = float(M[idx['train']].astype(np.float32).std())
print(len(te), tr_mean, tr_std)

In [ ]:
# M2 exactly as in its notebook, loaded from the saved checkpoint.
class M2CNN(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(blk(1,32), blk(32,64), blk(64,128), blk(128,256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(256,1))
    def forward(self, x): return self.head(self.features(x))

model = M2CNN().to(dev)
model.load_state_dict(torch.load(f'{MELS_DIR}/m2_best.pt', map_location=dev))
model.eval()

# Hook the last ReLU of the last block: the deepest map that still has
# spatial resolution (16 freq x 125 time) before the final max-pool.
target = model.features[3][5]
store = {}
target.register_forward_hook(lambda m, i, o: store.__setitem__('act', o))
target.register_full_backward_hook(lambda m, gi, go: store.__setitem__('grad', go[0]))

def gradcam(xb):
    """Standard Grad-CAM: weight each channel by its spatially averaged
    gradient, sum, ReLU. Returns maps at feature resolution and the scores."""
    xb = xb.to(dev)
    model.zero_grad(set_to_none=True)
    logit = model(xb).squeeze(1)
    logit.sum().backward()                       # each clip's own logit; no cross-talk
    A, G = store['act'], store['grad']           # (B, 256, 16, 125)
    w = G.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * A).sum(1))             # (B, 16, 125)
    return cam.detach().cpu().numpy(), torch.sigmoid(logit).detach().cpu().numpy()

cams, scores = [], []
for i in range(0, len(te), 32):
    x = (np.asarray(M[te[i:i+32]], dtype=np.float32) - tr_mean) / tr_std
    c, s = gradcam(torch.from_numpy(x).unsqueeze(1))
    cams.append(c); scores.append(s)
cams, scores = np.concatenate(cams), np.concatenate(scores)
pred = (scores > 0.5).astype(int)
from sklearn.metrics import balanced_accuracy_score
ba = balanced_accuracy_score(y_te, pred)
print('cams', cams.shape, '| balanced accuracy in fp32 (should be ~0.9732):', round(ba, 4))
assert abs(ba - 0.9732) < 3e-3

In [ ]:
# Frequency profile: normalise each clip's CAM to sum 1, then average over
# time -> share of the clip's evidence in each of the 16 frequency rows.
# Each row covers 8 mel bins; report the Hz range from librosa's own mel grid.
mel_hz = librosa.mel_frequencies(n_mels=128 + 2, fmin=0, fmax=8000)
row_lo = [float(mel_hz[8 * r]) for r in range(16)]
row_hi = [float(mel_hz[8 * r + 9]) for r in range(16)]
norm = cams / (cams.sum(axis=(1, 2), keepdims=True) + 1e-12)
prof = norm.sum(axis=2)                          # (N, 16), rows sum to 1

groups = {'real': y_te == 0, 'synthetic (all)': y_te == 1}
for g in sorted(set(gen_te[y_te == 1])):
    groups[g] = gen_te == g
# Only correctly classified clips: the explanation of a wrong call is not
# what we are trying to characterise.
correct = pred == y_te
profiles = {g: prof[m & correct].mean(0) for g, m in groups.items()}
top2 = {g: float(p[-2:].sum()) for g, p in profiles.items()}  # share in the top 2 rows
print('uniform share for 2 of 16 rows = 0.125')
for g in profiles:
    print(f'{g:18s} n={int((groups[g] & correct).sum()):4d}  share in top 2 rows '
          f'({row_lo[14]:.0f}-{row_hi[15]:.0f} Hz): {top2[g]:.3f}  | peak row {int(profiles[g].argmax())}')

In [ ]:
# Peakiness test for ref. 9: does the time-averaged CAM concentrate in a few
# narrow rows (stripes) or spread smoothly? Gini-style concentration of the
# per-clip frequency profile, and how consistent the peak row is within a
# generator (fraction of clips whose argmax row equals the generator's mode).
def concentration(p):                            # 0 = uniform, -> 1 = one row
    q = np.sort(p)[::-1]
    return float(q[:2].sum())
peak_rows = prof.argmax(1)
stripe = {}
for g, m in groups.items():
    mm = m & correct
    rows_g = peak_rows[mm]
    mode = int(np.bincount(rows_g, minlength=16).argmax())
    stripe[g] = {"n": int(mm.sum()),
                 "mean_top2_row_share": round(float(np.mean([concentration(p) for p in prof[mm]])), 4),
                 "peak_row_mode": mode,
                 "peak_row_mode_hz": [round(row_lo[mode]), round(row_hi[mode])],
                 "frac_clips_peak_at_mode": round(float((rows_g == mode).mean()), 4),
                 "share_top2_freq_rows": round(top2[g], 4),
                 "profile": [round(float(v), 4) for v in profiles[g]]}
    print(g, {k: v for k, v in stripe[g].items() if k != 'profile'})

res = {"model": "M2_CNN", "checkpoint": "fmc-m2-mels/m2_best.pt", "split": "test",
       "n_test": int(len(te)), "seed": 42, "fp32_balanced_accuracy": round(float(ba), 4),
       "target_layer": "features[3][5] (last ReLU before final max-pool), 16 x 125 map",
       "row_hz": [[round(a), round(b)] for a, b in zip(row_lo, row_hi)],
       "uniform_share_top2_rows": 0.125, "groups": stripe}
json.dump(res, open(f'{OUT}/gradcam_m2_results.json', 'w'), indent=2)

In [ ]:
# Figures: (1) mean frequency profile per group; (2) mean CAM per group over
# the spectrogram; (3) four individual correctly-classified synthetic clips.
fig, ax = plt.subplots(figsize=(6.5, 4))
for g, p in profiles.items():
    if g == 'synthetic (all)': continue
    ax.plot(p, range(16), 'o-' if g == 'real' else '.-', lw=2.5 if g == 'real' else 1.2,
            color='k' if g == 'real' else None, label=g)
ax.axvline(1 / 16, color='grey', ls='--', lw=.8, label='uniform')
ax.set_yticks(range(0, 16, 3))
ax.set_yticklabels([f'{row_lo[r]/1000:.1f}' for r in range(0, 16, 3)])
ax.set_ylabel('frequency row, lower edge (kHz)'); ax.set_xlabel('share of Grad-CAM mass')
ax.set_title('M2 Grad-CAM: where the evidence sits in frequency'); ax.legend(fontsize=7); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/gradcam_m2_freq_profile.png', dpi=150); plt.show()

names = ['real'] + sorted(set(gen_te[y_te == 1]))
fig, axes = plt.subplots(2, 3, figsize=(13, 5.5))
for ax, g in zip(axes.flat, names):
    mm = groups[g] & correct
    ax.imshow(norm[mm].mean(0), origin='lower', aspect='auto', cmap='magma',
              extent=[0, 10, 0, 16])
    ax.set_yticks([0, 4, 8, 12, 16]); ax.set_yticklabels([f'{mel_hz[min(8*r,129)]/1000:.1f}' for r in [0, 4, 8, 12, 16]])
    ax.set_title(f'{g} (n={int(mm.sum())})'); ax.set_xlabel('time (s)'); ax.set_ylabel('kHz')
plt.suptitle('M2 Grad-CAM averaged over correctly classified test clips')
plt.tight_layout(); plt.savefig(f'{OUT}/gradcam_m2_mean_maps.png', dpi=150); plt.show()

ex = [int(np.where((gen_te == g) & correct)[0][0]) for g in names[1:5]]
fig, axes = plt.subplots(2, 4, figsize=(15, 5.5))
for k, j in enumerate(ex):
    spec = np.asarray(M[te[j]], dtype=np.float32)
    up = np.kron(cams[j], np.ones((8, 8)))[:, :1001]
    axes[0, k].imshow(spec, origin='lower', aspect='auto', cmap='viridis')
    axes[0, k].set_title(f'{gen_te[j]}  p(synth)={scores[j]:.2f}')
    axes[1, k].imshow(spec, origin='lower', aspect='auto', cmap='gray')
    axes[1, k].imshow(up, origin='lower', aspect='auto', cmap='jet', alpha=.45)
    for a in axes[:, k]: a.set_xticks([]); a.set_ylabel('mel bin')
plt.tight_layout(); plt.savefig(f'{OUT}/gradcam_m2_examples.png', dpi=150); plt.show()
print(sorted(os.listdir(OUT)))

## Follow-up: is the Grad-CAM band a set of narrow spectral peaks?

The CAM rows are ~1 kHz tall, too coarse to see the narrow peaks ref. 9
describes. This cell looks at the signal itself: the long-term average
power spectrum at 3.9 Hz resolution (n_fft 4096) over 60 test clips per
group, from raw audio. Narrow peaks that recur at the same frequency in a
generator's clips would show up as spikes in that generator's average and
not in the real class's average. Descriptive only; no model involved.

In [ ]:
import soundfile as sf, time, io
from scipy.signal import welch
RAW_DIR  = find_input('fakemusiccaps-raw', 'mustango')
REAL_DIR = find_input('musiccaps-real-16k', '-0Gj8-vB1q4.wav')
# Paths built from the ytid and generator, not from subset.csv's stored path:
# v2-v5 failed because that stored path names the mount used in October, and
# this kernel's mount of musiccaps-real-16k is at a different location.
print('REAL_DIR', REAL_DIR, '| RAW_DIR', RAW_DIR)
paths = np.array([f'{REAL_DIR}/{y}.wav' if g == 'real' else f'{RAW_DIR}/{g}/{y}.wav'
                  for y, g in zip(sub.ytid, sub.generator)])
assert all(os.path.exists(paths[j]) for j in te), 'test-split audio missing'
rng = np.random.default_rng(42)
ltas = {}
for g in names:
    rows_g = te[gen_te == g]
    pick = rng.choice(rows_g, size=60, replace=False)
    P = []
    for j in pick:
        # v2-v4 failed here with libsndfile's opaque "System error" on the
        # first file read, a file every earlier kernel read fine. Reading the
        # bytes with Python first gives either the data or a real errno.
        with open(paths[j], 'rb') as fh:
            y, sr = sf.read(io.BytesIO(fh.read()), dtype='float32')
        f, p = welch(y[:160000], fs=16000, nperseg=4096)
        P.append(p / (p.sum() + 1e-20))             # each clip normalised: shape, not loudness
    ltas[g] = 10 * np.log10(np.mean(P, 0) + 1e-20)

# Peakiness above 3 kHz: how far the spectrum sticks out of its own 200 Hz
# running median (dB). Narrow, generator-specific peaks give large values.
from scipy.ndimage import median_filter
band = f >= 3000
peak_stats = {}
for g in names:
    resid = ltas[g] - median_filter(ltas[g], size=51)
    top = np.argsort(resid[band])[::-1][:5]
    peak_stats[g] = {"max_peak_db_above_local_median": round(float(resid[band].max()), 2),
                     "top5_peak_hz": [round(float(f[band][k])) for k in top],
                     "mean_db_4_6khz_minus_2_4khz": round(float(ltas[g][(f >= 4000) & (f < 6000)].mean()
                                                          - ltas[g][(f >= 2000) & (f < 4000)].mean()), 2)}
    print(g, peak_stats[g])
res["ltas_peaks"] = peak_stats
json.dump(res, open(f'{OUT}/gradcam_m2_results.json', 'w'), indent=2)

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
for g in names:
    axes[0].plot(f / 1000, ltas[g], lw=2 if g == 'real' else 1, color='k' if g == 'real' else None, label=g)
    if g != 'real':
        axes[1].plot(f / 1000, ltas[g] - ltas['real'], lw=1, label=g)
axes[0].set_ylabel('long-term avg power (dB, per-clip normalised)'); axes[0].legend(fontsize=7)
axes[1].set_ylabel('generator minus real (dB)'); axes[1].set_xlabel('frequency (kHz)')
for a in axes:
    a.axvspan(row_lo[13] / 1000, row_hi[14] / 1000, color='orange', alpha=.15)
    a.grid(alpha=.3)
axes[0].set_title('Long-term average spectrum, 60 test clips per group (shaded: M2 Grad-CAM peak band)')
plt.tight_layout(); plt.savefig(f'{OUT}/ltas_by_generator.png', dpi=150); plt.show()